In [9]:
import os
print("CWD:", os.getcwd())
print("exists ../../data:", os.path.exists("../../data"))
print("exists ../../data/train_processed.csv:", os.path.exists("../../data/train_processed.csv"))

CWD: c:\Users\ghkdr\OneDrive\바탕 화면\이스트캠프\데이콘\credit_default\notebooks\v2
exists ../../data: True
exists ../../data/train_processed.csv: True


In [11]:
import pandas as pd
from pandas.api.types import is_numeric_dtype

TRAIN_PATH = "../../data/train_processed.csv"
TEST_PATH  = "../../data/test_processed.csv"

df_tr = pd.read_csv(TRAIN_PATH)
df_te = pd.read_csv(TEST_PATH)

# 중복 제거
df_tr = df_tr.drop_duplicates().reset_index(drop=True)
df_te = df_te.drop_duplicates().reset_index(drop=True)

print("after dedup train:", df_tr.shape)
print("after dedup test :", df_te.shape)

# 타겟 분리
y = df_tr["credit"]
X_tr = df_tr.drop(columns=["credit"])
X_te = df_te.copy()

after dedup train: (24811, 20)
after dedup test : (9644, 19)


In [12]:
num_cols = [c for c in X_tr.columns if is_numeric_dtype(X_tr[c])]
cat_cols = [c for c in X_tr.columns if c not in num_cols]

print("num cols:", num_cols)
print("cat cols:", cat_cols)

num cols: ['child_num', 'income_total', 'FLAG_MOBIL', 'work_phone', 'phone', 'email', 'family_size', 'begin_month', 'age', 'employment_years', 'income_per_person']
cat cols: ['gender', 'car', 'reality', 'income_type', 'edu_type', 'family_type', 'house_type', 'occyp_type']


In [13]:
def clip_iqr(s: pd.Series):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    return s.clip(lo, hi)

for c in num_cols:
    X_tr[c] = clip_iqr(X_tr[c])
    X_te[c] = clip_iqr(X_te[c])

    med = X_tr[c].median()
    X_tr[c] = X_tr[c].fillna(med)
    X_te[c] = X_te[c].fillna(med)

In [14]:
from sklearn.preprocessing import OrdinalEncoder

for c in cat_cols:
    X_tr[c] = X_tr[c].fillna("__MISSING__").astype(str)
    X_te[c] = X_te[c].fillna("__MISSING__").astype(str)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols])
X_te[cat_cols] = encoder.transform(X_te[cat_cols])